In [24]:
import pandas as pd
""
df = pd.read_csv('forest_reserve_state.csv')

df.head()

,date,state,area
0,2003-01-01,Johor,356922.0
1,2003-01-01,Kedah,344530.0
2,2003-01-01,Kelantan,629687.0
3,2003-01-01,Melaka,5468.0
4,2003-01-01,Negeri Sembilan,165639.0


In [25]:
# Convert date column to datetime & extract year
df['year'] = pd.to_datetime(df['date']).dt.year

# 🔹 Clean data first
df = df[df['state'] != 'Semenanjung Malaysia']

rename_map = {
    'W.P. Kuala Lumpur': 'Kuala Lumpur',
    'W.P. Labuan': 'Labuan',
    'W.P. Putrajaya': 'Putrajaya'
}
df['state'] = df['state'].replace(rename_map)

# (Optional) Reset index
df = df.reset_index(drop=True)

# 🔹 Now find latest year
latest_year = df['year'].max()

# 🔹 Filter for the latest year
df_latest = df[df['year'] == latest_year]

print("Latest year:", latest_year)
print(df_latest.head())

Latest year: 2021
           date            state      area  year
288  2021-01-01            Johor  334502.0  2021
289  2021-01-01            Kedah  341976.0  2021
290  2021-01-01         Kelantan  629881.0  2021
291  2021-01-01           Melaka    5199.0  2021
292  2021-01-01  Negeri Sembilan  155143.0  2021


In [26]:
df_latest.to_csv("forest_reserve_cleaned.csv", index=False)

In [1]:
import pandas as pd 

df = pd.read_csv("air_pollution.csv", parse_dates=["date"])

print(df.head())

        date pollutant  concentration
0 2017-01-01        CO          0.561
1 2017-02-01        CO          0.530
2 2017-03-01        CO          0.589
3 2017-04-01        CO          0.662
4 2017-05-01        CO            NaN


In [2]:
# Extract year and month
df['year'] = df['date'].dt.year

# make sure concentration is numeric (empty strings -> NaN)
df['concentration'] = pd.to_numeric(df['concentration'], errors='coerce')

df = df[df['year'].isin([2019, 2020, 2021])]

# drop missing concentration values 
df = df.dropna(subset=['concentration'])

# group by pollutant + year, compute average
agg = df.groupby(["pollutant", "year"], as_index=False)["concentration"].mean()

# build hierarchical structure
rows = []
rows.append({'id': 'All', 'parent': '', 'value': ''})   # root -> parent empty cell

for pollutant in sorted(agg['pollutant'].unique()):
    rows.append({'id': pollutant, 'parent': 'All', 'value': ''})
    subset = agg[agg['pollutant'] == pollutant].sort_values('year')
    for _, r in subset.iterrows():
        rows.append({
            'id': f"{pollutant}-{int(r['year'])}",
            'parent': pollutant,
            'value': round(float(r['concentration']), 6)
        })

out = pd.DataFrame(rows, columns=['id','parent','value'])

In [3]:
out.head(30)

,id,parent,value
0,All,,
1,CO,All,
2,CO-2019,CO,0.650083
3,CO-2020,CO,0.544917
4,CO-2021,CO,0.53375
5,NO2,All,
6,NO2-2019,NO2,0.007192
7,NO2-2020,NO2,0.005767
8,NO2-2021,NO2,0.005692
9,O3,All,


In [4]:
out.to_csv("pollutants_cleaned.csv", index=False)
print("Cleaned file saved as pollutants_cleaned.csv")

Cleaned file saved as pollutants_cleaned.csv


In [2]:
import pandas as pd

# Load rainfall dataset
rainfall = pd.read_csv("mean_temp_rainfall.csv")

# Load pollution dataset
pollution = pd.read_csv("air_pollution.csv")

print(rainfall.head())
print(pollution.head())

      State Selected meteorological station   \
0     Johor                            Senai   
1     Johor                           Kluang   
2     Kedah                       Alor Setar   
3     Kedah                   Pulau Langkawi   
4  Kelantan                       Kota Bharu   

  Height above mean sea level in metres  Year  \
0                               (37.8m)  2000   
1                               (88.1m)  2000   
2                                (3.9m)  2000   
3                                (6.4m)  2000   
4                                (4.4m)  2000   

  Minimum Mean temperature in Celcius Maximum Mean temperature in Celcius  \
0                                22.9                                32.3   
1                                23.1                                32.0   
2                                23.6                                32.5   
3                                25.0                                32.0   
4                              

In [4]:
### rainfall data
# Force rainfall to numeric (remove text, commas, etc. if any)
rainfall["Total Rainfall in millimetres"] = pd.to_numeric(
    rainfall["Total Rainfall in millimetres"], errors="coerce"
)

# Keep relevant columns
rainfall_clean = rainfall[["State", "Year", "Total Rainfall in millimetres"]]

# Average rainfall per state per year (since some states have multiple stations)
rainfall_yearly = rainfall_clean.groupby(["State", "Year"], as_index=False).mean(numeric_only=True)

rainfall_yearly.rename(columns={"Total Rainfall in millimetres": "Rainfall_mm"}, inplace=True)

### pollution data
# Extract year from date
pollution["Year"] = pd.to_datetime(pollution["date"]).dt.year

# Remove 2017
pollution = pollution[pollution["Year"] != 2017]

# Average pollution per year
pollution_yearly = pollution.groupby("Year", as_index=False)["concentration"].mean()

# Merge by year
merged = pd.merge(rainfall_yearly, pollution_yearly, on="Year", how="inner")

merged.head(20)

,State,Year,Rainfall_mm,concentration
0,Johor,2018,2266.525000,6.974543
1,Johor,2019,2174.525000,8.387444
2,Johor,2020,2458.800000,5.431035
3,Johor,2021,2635.200000,5.668204
4,Kedah,2018,2202.200000,6.974543
5,Kedah,2019,2296.950000,8.387444
6,Kedah,2020,2570.000000,5.431035
7,Kedah,2021,2272.000000,5.668204
8,Kelantan,2018,2592.100000,6.974543
9,Kelantan,2019,1758.066667,8.387444


In [5]:
# Create categories based on tertiles
merged["PollutionCategory"] = pd.qcut(merged["concentration"], q=3, labels=["Low", "Medium", "High"])

merged.head(20)

,State,Year,Rainfall_mm,concentration,PollutionCategory
0,Johor,2018,2266.525000,6.974543,Medium
1,Johor,2019,2174.525000,8.387444,High
2,Johor,2020,2458.800000,5.431035,Low
3,Johor,2021,2635.200000,5.668204,Low
4,Kedah,2018,2202.200000,6.974543,Medium
5,Kedah,2019,2296.950000,8.387444,High
6,Kedah,2020,2570.000000,5.431035,Low
7,Kedah,2021,2272.000000,5.668204,Low
8,Kelantan,2018,2592.100000,6.974543,Medium
9,Kelantan,2019,1758.066667,8.387444,High


In [6]:
merged.to_csv("rainfall_pollution_ready.csv", index=False)

In [12]:
import pandas as pd 

pollution = pd.read_csv("air_pollution_station.csv")
temp = pd.read_csv("mean_temp_rainfall.csv")

# Clean and preprocess
pollution.columns = pollution.columns.str.strip()
temp.columns = temp.columns.str.strip()

pollution["AvgPollution"] = (pollution["Maximum"] + pollution["Minimum"]) / 2
temp["MeanTemp"] = (pd.to_numeric(temp["Minimum Mean temperature in Celcius"], errors="coerce") +
                    pd.to_numeric(temp["Maximum Mean temperature in Celcius"], errors="coerce")) / 2

# Clean text
pollution["Selected Stations"] = pollution["Selected Stations"].str.strip().str.lower()
temp["Selected meteorological station"] = temp["Selected meteorological station"].str.strip().str.lower()

# Manual mapping between pollution stations and meteorological stations
station_map = {
    "bandaraya melaka, melaka": "melaka",
    "cheras, kuala lumpur": None,  # no corresponding meteorological station
    "kota kinabalu, sabah": None,  # not in temp dataset
    "kuching, sarawak": None,
    "larkin, johor bahru": "senai",
    "miri, sarawak": None,
    "seberang jaya, pulau pinang": "butterworth"
}

# Apply mapping
pollution["MappedStation"] = pollution["Selected Stations"].map(station_map)

# Drop stations that can’t be mapped
pollution = pollution.dropna(subset=["MappedStation"])

# Merge using mapped names
merged = pd.merge(
    pollution,
    temp,
    left_on=["Year", "MappedStation"],
    right_on=["Year", "Selected meteorological station"],
    how="inner"
)

# Select columns
merged = merged[["Year", "Selected Stations", "AvgPollution", "MeanTemp"]]

# Temperature bins for visualization
bins = [15, 20, 22, 24, 25, 26, 27, 28, 29, 30, 32, 34]
labels = ['15–20', '20–22', '22–24', '24–25', '25–26', '26–27', '27–28', '28–29', '29–30', '30–32', '32–34']
merged["TempRange"] = pd.cut(merged["MeanTemp"], bins=bins, labels=labels, include_lowest=True)

# Save result
merged.to_csv("pollution_temp_heatmap.csv", index=False)

print(merged)


    Year            Selected Stations  AvgPollution  MeanTemp TempRange
0   2000     bandaraya melaka, melaka          63.0     27.95     27–28
1   2000          larkin, johor bahru          63.0     27.60     27–28
2   2000  seberang jaya, pulau pinang          65.5     27.80     27–28
3   2001     bandaraya melaka, melaka          65.0     27.90     27–28
4   2001          larkin, johor bahru          64.5     27.50     27–28
..   ...                          ...           ...       ...       ...
58  2019          larkin, johor bahru          98.5     28.35     28–29
59  2019  seberang jaya, pulau pinang         109.5     28.55     28–29
60  2020     bandaraya melaka, melaka          59.0     28.15     28–29
61  2020          larkin, johor bahru          57.0     28.30     28–29
62  2020  seberang jaya, pulau pinang          60.0     28.45     28–29

[63 rows x 5 columns]
